# Module 4: Search Evaluation



In [28]:
import pandas as pd

In [29]:
filepath = "./data/ground_truth.csv"

In [30]:
df_ground_truth = pd.read_csv(filepath)

In [31]:
df_ground_truth.head()

,question,document
0,Can I still join the course if I'm late to it?,74eb249bbf
1,"If I join the course now, can I still get a ce...",74eb249bbf
2,Do I need to finish the project before submiss...,74eb249bbf
3,"Is it too late to start this course, or can I ...",74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [32]:
ground_truth = df_ground_truth.to_dict(orient="records")

Now that we have our search data ready, we need to index our documents so that we will be able to search them.

In [33]:
from ingest import load_faq_data, build_index

In [34]:
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [35]:
def text_search(query:str):
    boost_dict = {"question": 3.0, "section": 0.5}
    # If we were using all docs we could also define a filter dict here to add to the search that is returned.
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [36]:
from minsearch import Index

In [37]:
ground_truth[0]

{'question': "Can I still join the course if I'm late to it?",
 'document': '74eb249bbf'}

In [38]:
q = ground_truth[0]["question"]

In [39]:
q

"Can I still join the course if I'm late to it?"

In [40]:
results = text_search(q)

In [41]:
results[0]["id"]

'74eb249bbf'

In [42]:
for res in results:
    if res["id"] == ground_truth[0]["document"]:
        print(res)

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [43]:
q = ground_truth[0]
q

{'question': "Can I still join the course if I'm late to it?",
 'document': '74eb249bbf'}

In [44]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [45]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False


In [46]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

In [47]:
relevance

[1, 0, 0, 0, 0]

## Search across all search terms

In [48]:
def compute_relevant_text(q: dict[str]):
    # get id of document used to generate the query
    doc_id = q["document"]
    # search q* term against original docs
    results = text_search(query=q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [49]:
from tqdm.auto import tqdm

In [50]:
def compute_relevance_total_text(ground_truth: list[dict]):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevant_text(q)
        relevance_total.append(relevance)
    
    return relevance_total

In [51]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/515 [00:00<?, ?it/s]

In [52]:
relevance

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

Update the relevance functions so that we can use any search function, not just search.

In [53]:
def compute_relevant(q: dict[str],search_function: Callable[[str], list[dict[str]]]) -> list[int]:
    # get id of document used to generate the query
    doc_id = q["document"]
    # search q* term against original docs
    results = search_function(query=q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [54]:
def compute_relevance_total(ground_truth: list[dict], search_function: Callable[[str], list[dict[str]]]) -> list[list[int]]:
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevant(q, search_function)
        relevance_total.append(relevance)
    
    return relevance_total

## Search Evaluation Metrics


In [55]:
number_of_hits = 0

for q in relevance:
    if 1 in q:
        number_of_hits += 1

number_of_hits

445

In [57]:
hit_rate = number_of_hits/ len(relevance)

hit_rate

0.8640776699029126

In [58]:
def hit_rate(relevance: list[list[int]]) -> int:
    number_of_hits = 0

    for q_results in relevance:
        if 1 in q_results:
            number_of_hits += 1
    
    hr = number_of_hits / len(relevance)
    
    return hr

In [61]:
total_score = 0.0

for q in relevance:
    for rank in range(len(q)):
        if q[rank] == 1:
            score = 1 / (rank + 1)
            total_score += score
            break

total_score

382.3333333333332

In [62]:
def mrr(relevance: list[list[int]]) -> int:
    total_score = 0.0

    for q in relevance:
        for rank in range(len(q)):
            if q[rank] == 1:
                score = 1 / (rank + 1)
                total_score += score
                break

    mrr = total_score/len(relevance)
    return mrr

In [63]:
mrr(relevance)

0.7423948220064722

In [64]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [65]:
evaluate(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8640776699029126, 'mrr': 0.7423948220064722}

Now we have this function we can use it to evaluate any other search method by just updating the search function to be used in the function call arguments.